## 1. Load the house data

Read the CSV file. Each row is a house with 8 inputs and its price.
We will predict the price.

In [2]:
import pandas as pd
data = pd.read_csv("../house_data.csv")

In [3]:
data.head()

,sqft,bedrooms,bathrooms,age,floors,garage,lot_size,city_index,price
0,1360,2,4,18,1,0,5566,1,248990
1,4272,3,4,29,2,2,2909,0,941769
2,3592,1,4,13,2,1,7364,8,378015
3,966,1,4,19,1,0,3581,0,196887
4,4926,2,4,48,1,1,1691,3,1063441


In [4]:
type(data)

pandas.DataFrame

## 2. Separate the inputs (X) and the output (Y)

Y is the price. X is the 8 other columns.
Check the shapes: 1000 rows, 8 inputs, 1 output.

In [5]:
X = data.drop(columns=['price'])

In [6]:
Y = data[['price']]

In [7]:
X.shape

(1000, 8)

In [8]:
Y.shape

(1000, 1)

In [9]:
type(X)

pandas.DataFrame

## 3. Convert to NumPy float32

PyTorch works with float32 numbers, so convert the tables first.
to_numpy() turns a DataFrame into an array.

In [11]:
import numpy as np

In [12]:
X_new = X.to_numpy(dtype=np.float32)
Y_new = Y.to_numpy(dtype=np.float32)

In [13]:
X_new

array([[1.360e+03, 2.000e+00, 4.000e+00, ..., 0.000e+00, 5.566e+03,
        1.000e+00],
       [4.272e+03, 3.000e+00, 4.000e+00, ..., 2.000e+00, 2.909e+03,
        0.000e+00],
       [3.592e+03, 1.000e+00, 4.000e+00, ..., 1.000e+00, 7.364e+03,
        8.000e+00],
       ...,
       [2.606e+03, 4.000e+00, 4.000e+00, ..., 1.000e+00, 1.495e+03,
        0.000e+00],
       [4.723e+03, 5.000e+00, 2.000e+00, ..., 0.000e+00, 3.148e+03,
        9.000e+00],
       [3.268e+03, 4.000e+00, 3.000e+00, ..., 2.000e+00, 1.556e+03,
        2.000e+00]], shape=(1000, 8), dtype=float32)

In [14]:
X_new.dtype

dtype('float32')

## 4. Split into training and test data

Use 80% of the houses for training and keep 20% for testing.
random_state=42 makes the split repeatable.

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
x_train, x_test, y_train, y_test = train_test_split(X_new, Y_new, test_size=0.2, random_state=42)

## 5. Scale the numbers (StandardScaler)

Features such as sqft and bedrooms are on very different scales, which makes training slow and unstable.
Fit the scaler on the training data only, then transform both parts.

In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
x_scalar = StandardScaler().fit(x_train)
y_scalar = StandardScaler().fit(y_train)

In [19]:
x_scalar

StandardScaler()

In [20]:
x_train_scalar = x_scalar.transform(x_train).astype(np.float32)
x_test_scalar = x_scalar.transform(x_test).astype(np.float32)
y_train_scalar = y_scalar.transform(y_train).astype(np.float32)
y_test_scalar = y_scalar.transform(y_test).astype(np.float32)

In [21]:
x_train_scalar

array([[-0.6277223 , -0.6819861 , -0.44865718, ..., -0.04929634,
        -0.06389936, -1.5591687 ],
       [ 0.56318504,  1.4299709 , -1.3240858 , ...,  1.1831132 ,
         0.6206979 ,  0.48285297],
       [ 1.3523309 , -0.6819861 , -0.44865718, ...,  1.1831132 ,
        -1.475833  ,  1.1635269 ],
       ...,
       [-1.0622318 ,  1.4299709 ,  0.4267714 , ..., -0.04929634,
        -1.2116971 , -1.2188318 ],
       [-0.7579154 ,  0.7259852 ,  0.4267714 , ..., -1.281706  ,
         0.7054062 , -1.2188318 ],
       [-1.1588782 ,  0.7259852 , -1.3240858 , ..., -1.281706  ,
        -1.170113  ,  0.8231899 ]], shape=(800, 8), dtype=float32)

## 6. Install and import PyTorch

PyTorch is the library for building neural networks.
Install it once with pip.

In [22]:
!python3 -m pip install torch


  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.5/138.5 MB 94.5 MB/s  0:00:01m0:00:0100:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]32m5/6 [torch]kx]

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [23]:
import torch

## 7. Convert the arrays to tensors

A tensor is the PyTorch version of an array.
The model only accepts tensors.

In [24]:
x_train_tensor = torch.from_numpy(x_train_scalar)
x_test_tensor = torch.from_numpy(x_test_scalar)
y_train_tensor = torch.from_numpy(y_train_scalar)
y_test_tensor= torch.from_numpy(y_test_scalar)

In [25]:
x_train_tensor

tensor([[-0.6277, -0.6820, -0.4487,  ..., -0.0493, -0.0639, -1.5592],
        [ 0.5632,  1.4300, -1.3241,  ...,  1.1831,  0.6207,  0.4829],
        [ 1.3523, -0.6820, -0.4487,  ...,  1.1831, -1.4758,  1.1635],
        ...,
        [-1.0622,  1.4300,  0.4268,  ..., -0.0493, -1.2117, -1.2188],
        [-0.7579,  0.7260,  0.4268,  ..., -1.2817,  0.7054, -1.2188],
        [-1.1589,  0.7260, -1.3241,  ..., -1.2817, -1.1701,  0.8232]])

In [26]:
x_train_tensor.shape

torch.Size([800, 8])

## 8. Define the neural network (no hidden layer)

One Linear(8, 1) layer takes the 8 inputs and gives 1 output, the price.
forward() says how the data flows through the model.

In [28]:
import torch.nn as nn
import torch.optim as optim

In [29]:
torch.manual_seed(42)

class HousePricePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8,1),
        )
    def forward(self, x_train_data):
        return self.network(x_train_data)

model = HousePricePredictor()

In [30]:
model

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [31]:
for param in model.parameters():
    print (param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


## 9. Choose the loss function

MSELoss measures how far the predictions are from the real prices.
Training tries to make this number small.

In [32]:
criterion = nn.MSELoss()

## 10. Choose the optimizer

Adam updates the weights to reduce the loss.
lr=0.001 is the learning rate, the size of each update.

In [33]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 11. Train the model

Each epoch does a forward pass, calculates the loss, goes backward and updates the weights.
The loss should go down as it learns.

In [34]:
model.train()

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=1, bias=True)
  )
)

In [35]:
epochs = 1000

In [36]:
for epoch in range(1, epochs+1):
    #1. Forward Pass
    y_predictions = model(x_train_tensor)
    #2. Calculate loss
    loss = criterion(y_predictions, y_train_tensor )
    #3. Optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{epochs} | loss: {loss.item():.6f}")
    

Epoch  100/1000 | loss: 0.646355
Epoch  200/1000 | loss: 0.468932
Epoch  300/1000 | loss: 0.369166
Epoch  400/1000 | loss: 0.311933
Epoch  500/1000 | loss: 0.278131
Epoch  600/1000 | loss: 0.257810
Epoch  700/1000 | loss: 0.245684
Epoch  800/1000 | loss: 0.238676
Epoch  900/1000 | loss: 0.234819
Epoch 1000/1000 | loss: 0.232816


## 12. Look at the learned weights

These are the numbers the network learned.
Compare them with the random weights printed before training.

In [37]:
for param in model.parameters():
    print (param)

Parameter containing:
tensor([[ 0.8390,  0.0394,  0.0532,  0.0016, -0.0099,  0.0109, -0.0161, -0.0149]],
       requires_grad=True)
Parameter containing:
tensor([0.0002], requires_grad=True)


## 13. Predict on the test data

eval() and no_grad() switch off training behaviour.
The predictions are still scaled, so convert them back to dollars next.

In [38]:
model.eval()
with torch.no_grad():
    y_pred_test = model(x_test_tensor)
    y_pre_numpy = y_pred_test.numpy()
y_pre_numpy

array([[ 0.8235236 ],
       [-0.261167  ],
       [ 1.2680699 ],
       [ 1.3975035 ],
       [ 0.46973556],
       [ 0.51852065],
       [ 1.2882466 ],
       [ 0.9062751 ],
       [ 0.62757653],
       [ 0.9540573 ],
       [ 0.5121409 ],
       [ 0.23520918],
       [-1.3438824 ],
       [ 1.4557236 ],
       [-0.2395056 ],
       [-1.2773513 ],
       [-0.47504413],
       [-0.8039661 ],
       [ 0.37906718],
       [-0.85364705],
       [ 0.36686024],
       [-0.35852185],
       [ 0.2699199 ],
       [ 0.7896064 ],
       [-0.44410965],
       [-0.8232499 ],
       [-0.44570738],
       [-0.56300575],
       [-1.2949674 ],
       [-0.61360157],
       [-1.0893629 ],
       [ 0.97140193],
       [ 0.57652   ],
       [-0.9076676 ],
       [-0.4331271 ],
       [ 0.4769018 ],
       [-0.617337  ],
       [-0.2700478 ],
       [-1.144603  ],
       [-1.4081814 ],
       [-1.0323789 ],
       [-1.2573671 ],
       [-0.42475837],
       [-1.0286754 ],
       [ 1.5417632 ],
       [ 1

## 14. Convert predictions back to dollars

inverse_transform undoes the scaling we applied to the prices.
Now the predictions are real prices.

In [39]:
y_pred_test = y_scalar.inverse_transform(y_pre_numpy).flatten()
y_pred_test

array([627129.1 , 397125.97, 721393.  , 748838.75, 552110.2 , 562454.8 ,
       725671.4 , 644676.2 , 585579.56, 654808.2 , 561102.  , 502380.06,
       167541.62, 761184.  , 401719.16, 181649.22, 351774.4 , 282028.16,
       532884.44, 271493.56, 530296.  , 376482.38, 509740.28, 619937.2 ,
       358333.9 , 277939.12, 357995.12, 333122.6 , 177913.81, 322394.  ,
       221511.22, 658486.  , 574753.3 , 260038.78, 360662.7 , 553629.75,
       321601.94, 395242.84, 209797.81, 153907.34, 233594.38, 185886.75,
       362437.25, 234379.67, 779428.25, 691524.25, 600305.6 , 601472.7 ,
       708793.4 , 510873.66, 199182.61, 521197.7 , 580091.56, 526952.75,
       376058.2 , 587057.9 , 154917.6 , 219597.72, 513575.06, 173396.9 ,
       710781.6 , 274072.72, 199696.69, 357314.62, 442640.44, 432593.72,
       302190.25, 552599.56, 666383.  , 520749.47, 355981.5 , 644232.75,
       241989.11, 400917.34, 501887.62, 514260.8 , 362133.3 , 151608.4 ,
       423019.94, 654844.9 , 408931.44, 266766.62, 

## 15. Evaluate

R2 and the average error show how good the predictions are.
Compare a few actual and predicted prices side by side.

In [40]:
from sklearn.metrics import r2_score, mean_absolute_error

y_actual = y_test.flatten()
print("R2 score :", round(r2_score(y_actual, y_pred_test), 3))
print("Avg error: $", round(mean_absolute_error(y_actual, y_pred_test)))

R2 score : 0.764
Avg error: $ 86116


In [41]:
pd.DataFrame({"actual": y_actual[:8].round(), "predicted": y_pred_test[:8].round()})

,actual,predicted
0,746264.0,627129.0
1,349011.0,397126.0
2,853367.0,721393.0
3,784465.0,748839.0
4,342987.0,552110.0
5,563293.0,562455.0
6,688837.0,725671.0
7,849834.0,644676.0
